In [23]:
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
from google.colab import drive

# Isso vai abrir uma janelinha pedindo autorização para acessar seu Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
from google.colab import files
import io

uploaded = files.upload()

nome_arquivo_local = list(uploaded.keys())[0]
df_local = pd.read_excel(io.BytesIO(uploaded[nome_arquivo_local]))

print(f"Arquivo {nome_arquivo_local} carregado com sucesso do seu PC!")
df_local.head(20)

Saving Pesquisa_Mercado_Livre_2026-03-30_21-15-11.xlsx to Pesquisa_Mercado_Livre_2026-03-30_21-15-11 (1).xlsx
Arquivo Pesquisa_Mercado_Livre_2026-03-30_21-15-11 (1).xlsx carregado com sucesso do seu PC!


,Produto,Preço (R$),Frete (R$),Selo_Full,Link
0,Memória RAM preto/vermelho 8GB 1x8GB Patriot P...,496,0,False,https://click1.mercadolivre.com.br/mclics/clic...
1,Memória Ddr4 Corsair Vengeance Rgb Rs 32gb 2x1...,2180,0,False,https://click1.mercadolivre.com.br/mclics/clic...
2,Módulo De Memória De Desktop Ddr4 De 16 Gb/240...,1351,0,False,https://click1.mercadolivre.com.br/mclics/clic...
3,Memória Ram 16gb Ddr4 2666mhz Nexure Dominion ...,899,0,True,https://click1.mercadolivre.com.br/mclics/clic...
4,Memória Ram Para Pc Ddr4 16gb 3200mhz Cl16 (1x...,979,0,False,https://www.mercadolivre.com.br/memoria-ram-pa...
5,Memória RAM Premier cor verde 8GB 1 Adata AD4S...,398,0,False,https://www.mercadolivre.com.br/memoria-ram-pr...
6,Memória RAM Notebook Ddr4 3200mhz 16gb Cl22 co...,1029,0,True,https://www.mercadolivre.com.br/memoria-ram-no...
7,Memória RAM verde 4GB 1 Smart SH564128FJ8NWRNSQG,77,0,False,https://www.mercadolivre.com.br/memoria-ram-ve...
8,"Memória Mancer Dantalion Z, 8gb (1X8gb), DDR4,...",575,0,False,https://www.mercadolivre.com.br/memoria-mancer...
9,Memoria Crucial 8gb Ddr4 3200mhz - Cb8gu3200,508,0,False,https://www.mercadolivre.com.br/memoria-crucia...


In [26]:
# Criando a coluna de Custo Total
df_local['Custo_Total'] = df_local['Preço (R$)'] + df_local['Frete (R$)']

# Ordenando do mais barato para o mais caro
df_local = df_local.sort_values(by='Custo_Total').reset_index(drop=True)

print("Cálculos concluídos! Aqui estão as 5 ofertas mais baratas (Custo Total):")
df_local[['Produto', 'Custo_Total', 'Selo_Full']].head(5)

Cálculos concluídos! Aqui estão as 5 ofertas mais baratas (Custo Total):


,Produto,Custo_Total,Selo_Full
0,Blister Para Memória De Note Sodimm Ddr4 Kit C...,58,False
1,Memória RAM verde 4GB 1 Smart SH564128FJ8NWRNSQG,77,False
2,Embalagem Box Coletivo Para 25 Memórias Pc Kit...,129,False
3,Memória Bestbattery Para Notebook Ddr4 4gb 1.2...,149,False
4,Memória Ram Color Verde 4gb 1 Smart Sf464128ck...,170,True


In [30]:
# Filtrando apenas os que são FULL e pegando o "campeão" de preço
# Nota: Garantimos que o Selo_Full seja tratado como booleano (True/False)
melhores_ofertas_full = df_local[df_local['Selo_Full'] == True].head(3)

if not melhores_ofertas_full.empty:
    vencedor = melhores_ofertas_full.iloc[0]
    print(f"🏆 RECOMENDAÇÃO DA IA: \nO produto '{vencedor['Produto']}' é a melhor escolha.")
    print(f"Custo: R$ {vencedor['Custo_Total']:.2f} com entrega FULL.")
else:
    print("Nenhuma oferta com selo FULL encontrada para recomendação automática.")

🏆 RECOMENDAÇÃO DA IA: 
O produto 'Memória Ram Color Verde 4gb 1 Smart Sf464128ckhiwdfseg' é a melhor escolha.
Custo: R$ 170.00 com entrega FULL.


In [34]:
import pandas as pd
import plotly.express as px
import os
from IPython.display import HTML, display

# --- 1. PREPARAÇÃO DOS DADOS (Obrigatório para o gráfico existir) ---
df_local['Custo_Total'] = df_local['Preço (R$)'] + df_local['Frete (R$)']
df_local['Entrega Full ⚡'] = df_local['Selo_Full'].map({True: 'Sim', False: 'Não'})
df_local = df_local.sort_values(by='Custo_Total', ascending=True).reset_index(drop=True)

# Top 30 Únicos
df_30 = df_local.head(30).copy()
df_30['Nome_Curto'] = df_30['Produto'].str[:35] + "..."
df_plot = df_30.drop_duplicates(subset=['Nome_Curto']).copy()

# Lógica de Cores para o Ranking
cores_ranking = []
for i, row in df_plot.iterrows():
    if i == 0: cores_ranking.append('#FFD700')
    elif i == 1: cores_ranking.append('#C0C0C0')
    elif i == 2: cores_ranking.append('#CD7F32')
    else: cores_ranking.append('#10B981' if row['Selo_Full'] else '#475569')

# --- 2. CRIAÇÃO DOS GRÁFICOS (Definindo fig_bar e fig_pie) ---
max_v = df_plot['Custo_Total'].max()
fig_bar = px.bar(df_plot[::-1], x="Custo_Total", y="Nome_Curto", orientation='h', template="plotly_dark")
fig_bar.update_traces(
    marker_color=cores_ranking[::-1],
    texttemplate='R$ %{x:.2f}', textposition='outside',
    customdata=df_plot['Link'][::-1],
    cliponaxis=False
)
fig_bar.update_layout(
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showticklabels=False, showgrid=False, range=[0, max_v * 1.5]),
    yaxis=dict(title="", tickfont=dict(color="#ffffff", size=10)),
    margin=dict(l=0, r=100, t=20, b=20), height=1000, showlegend=False, bargap=0.3
)

fig_pie = px.pie(df_local, names='Entrega Full ⚡', hole=0.6,
                 color='Entrega Full ⚡',
                 color_discrete_map={'Sim': '#10B981', 'Não': '#475569'},
                 template="plotly_dark")
fig_pie.update_layout(showlegend=False, margin=dict(t=0, b=0, l=0, r=0), height=230)

# --- 3. MONTAGEM DO HTML (O Dashboard) ---
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs='cdn', div_id='bar_div')
pie_html = fig_pie.to_html(full_html=False, include_plotlyjs='cdn')

dashboard_completo_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {{ background-color: #020617; margin: 0; padding: 20px; font-family: 'Inter', sans-serif; color: white; display: flex; justify-content: center; }}
        .main-box {{ background: #020617; padding: 35px; border-radius: 20px; border: 1px solid #1e293b; max-width: 1150px; width: 100%; }}
        .kpi-row {{ display: flex; gap: 15px; margin-bottom: 25px; }}
        .kpi-card {{ flex: 1; background: #0f172a; padding: 15px 20px; border-radius: 12px; border: 1px solid #1e293b; }}
        .kpi-card span {{ color: #94a3b8; font-size: 10px; font-weight: bold; text-transform: uppercase; }}
        .kpi-card h2 {{ color: #ffffff; margin: 5px 0 0 0; font-size: 20px; }}
        .dot {{ width: 8px; height: 8px; border-radius: 50%; display: inline-block; margin-right: 5px; }}
        .container-flex {{ display: flex; gap: 20px; align-items: flex-start; }}
        .col-left {{ flex: 2; background: #0f172a; padding: 20px; border-radius: 15px; border: 1px solid #1e293b; }}
        .col-right {{ flex: 0.8; display: flex; flex-direction: column; gap: 20px; }}
        .ranking-scroll {{ height: 500px; overflow-y: auto; scrollbar-width: none; -ms-overflow-style: none; }}
        .ranking-scroll::-webkit-scrollbar {{ display: none; }}
    </style>
</head>
<body>
    <div class="main-box">
        <div style="margin-bottom: 30px; border-bottom: 1px solid #1e293b; padding-bottom: 20px;">
            <h1 style="color:white; margin:0;">Market Intelligence Dashboard</h1>
            <p style="color: #cbd5e1; font-size: 14px; margin-top: 5px;">📍 Otimização de Compras & Análise Competitiva</p>
        </div>
        <div class="kpi-row">
            <div class="kpi-card"><span>MELHOR PREÇO</span><h2>R$ {df_plot.iloc[0]['Custo_Total']:.2f}</h2></div>
            <div class="kpi-card"><span>MÉDIA DO SETOR</span><h2>R$ {df_local['Custo_Total'].mean():.2f}</h2></div>
            <div class="kpi-card"><span>PLATAFORMA</span><h2>Mercado Livre Brasil</h2></div>
        </div>
        <div class="container-flex">
            <div class="col-left">
                <h3 style="font-size: 15px; margin-bottom: 10px; color:white;">📊 Ranking Top 30</h3>
                <div class="ranking-scroll">{bar_html}</div>
            </div>
            <div class="col-right">
                <div style="background: #0f172a; padding: 20px; border-radius: 15px; border: 1px solid #1e293b; text-align: center;">
                    <h3 style="font-size: 15px; margin-bottom: 10px; color:white;">🚀 Breakdown Logístico</h3>
                    {pie_html}
                    <div style="display: flex; justify-content: center; gap: 15px; margin-top: 5px;">
                        <div style="font-size: 11px;"><span class="dot" style="background:#10B981;"></span>Sim (Full)</div>
                        <div style="font-size: 11px;"><span class="dot" style="background:#475569;"></span>Não</div>
                    </div>
                </div>
                <div style="background: #10B981; color: #020617; padding: 15px; border-radius: 12px; text-align: center; font-weight: bold; font-size: 13px;">
                    ⚡ Entrega Full Priorizada
                </div>
            </div>
        </div>
    </div>
    <script>
        window.onload = function() {{
            var plotDiv = document.getElementById('bar_div');
            if(plotDiv) {{
                plotDiv.on('plotly_click', function(data){{
                    var url = data.points[0].customdata;
                    if(url) window.open(url, '_blank');
                }});
            }}
        }};
    </script>
</body>
</html>
"""

# --- 4. SALVAMENTO E EXIBIÇÃO ---
caminho_final = '/content/drive/MyDrive/RPA_Resultados/DASHBOARD_PROFISSIONAL.html'
os.makedirs(os.path.dirname(caminho_final), exist_ok=True)
with open(caminho_final, 'w', encoding='utf-8') as f:
    f.write(dashboard_completo_html)

print(f"✅ Dashboard gerado e salvo com sucesso em: {caminho_final}")
display(HTML(dashboard_completo_html))

✅ Dashboard gerado e salvo com sucesso em: /content/drive/MyDrive/RPA_Resultados/DASHBOARD_PROFISSIONAL.html
